<a href="https://colab.research.google.com/github/RomaParakh/uae-car-depreciation-analysis/blob/main/2_Models_Car_Age.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [ ]:
dataset = pd.read_excel("/content/yallamotor_used_cars.xlsx","Sheet1", na_values=["N/A", "n/a", "NA", "na", "nan", ""])
df = dataset
df.columns = df.columns.str.strip()

In [ ]:
# Check missing values before dropping
print(f"Mileage NaN count (before): {df['mileage(km)'].isnull().sum()}")
print(f"Price NaN count (before): {df['price'].isnull().sum()}")
print(f"Total rows before dropping: {len(df)}")


Mileage NaN count (before): 175
Price NaN count (before): 39
Total rows before dropping: 1975


In [ ]:
# Drop rows with missing mileage_km OR price, directly from source
df = df.dropna(subset=["mileage(km)", "price"]).reset_index(drop=True)

In [ ]:
# Confirm the drop worked
print(f"\nMileage NaN count (after): {df['mileage(km)'].isnull().sum()} (expect 0)")
print(f"Price NaN count (after): {df['price'].isnull().sum()} (expect 0)")
print(f"Total rows after dropping: {len(df)}")


Mileage NaN count (after): 0 (expect 0)
Price NaN count (after): 0 (expect 0)
Total rows after dropping: 1797


In [ ]:
df1 = df.copy()

In [ ]:
df1.head()

,car_title,price,model_year,mileage(km),monthly_payment,specs,location,fuel_type,transmission,deal_rating,semiconductor_flag,market_period,car_age,financing_available,full_service_history,inspected,under_warranty,urgent_sale,export_options,helper,inflation,oil_price,conflict_index,age_inflation_interaction,age_conflict_interaction,age_semiconductor_interaction,mileage_oil_interaction
0,Used Renault Duster,51000.0,2025,25688.0,744,GCC Specs,Abu Dhabi,Petrol,CVT,Fair Deal,0,Post-Recovery,1,0,0,0,0,0,Not For Export,2025,1.60,69.14,100.000000,1.60,100.000000,0,1776068.32
1,Used Nissan Patrol,181300.0,2023,85122.0,2644,GCC Specs,Abu Dhabi,Petrol,Not Mentioned,Fair Deal,1,Recovery,3,0,0,0,0,0,Not For Export,2023,1.63,82.49,67.348798,4.89,202.046395,3,7021713.78
2,Used Nissan Patrol,200900.0,2024,66000.0,2930,GCC Specs,Abu Dhabi,Petrol,Not Mentioned,Good Deal,0,Post-Recovery,2,0,0,0,0,0,Not For Export,2024,1.66,80.52,100.000000,3.32,200.000000,0,5314320.00
3,Used Infiniti Q50,125000.0,2024,16.0,1823,GCC Specs,Abu Dhabi,Petrol,Not Mentioned,Fair Deal,0,Post-Recovery,2,0,0,0,0,0,Not For Export,2024,1.66,80.52,100.000000,3.32,200.000000,0,1288.32
4,Used Renault Megane,48000.0,2024,8961.0,700,GCC Specs,Abu Dhabi,Petrol,CVT,Great Deal,0,Post-Recovery,2,0,0,0,0,0,Not For Export,2024,1.66,80.52,100.000000,3.32,200.000000,0,721539.72


In [ ]:
df1.rename(columns={"car_age ": "car_age"}, inplace=True)

In [ ]:
df1.loc[df1["mileage(km)"].isna(), "mileage_oil_interaction"] = np.nan

In [ ]:
print(df1.shape)
print(df1.info())
print(df1.isnull().sum())

(1797, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1797 entries, 0 to 1796
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   car_title                      1797 non-null   object 
 1   price                          1797 non-null   float64
 2   model_year                     1797 non-null   int64  
 3   mileage(km)                    1797 non-null   float64
 4   monthly_payment                1797 non-null   object 
 5   specs                          1797 non-null   object 
 6   location                       1797 non-null   object 
 7   fuel_type                      1797 non-null   object 
 8   transmission                   1797 non-null   object 
 9   deal_rating                    1797 non-null   object 
 10  semiconductor_flag             1797 non-null   int64  
 11  market_period                  1797 non-null   object 
 12  car_age                        1797 n

In [ ]:
df1.columns = df1.columns.str.strip()  # removes leading/trailing spaces

In [ ]:
print(df1[df1["model_year"] < 2015]["conflict_index"].unique())
print(df1[df1["model_year"] == 2026]["conflict_index"].unique())

[0.]
[0.]


In [ ]:
# Only replace 0 with NaN for years where conflict data doesn't exist
no_conflict_data = df1["model_year"].isin([2011, 2012, 2013, 2014, 2026])
df1.loc[no_conflict_data, "conflict_index"] = np.nan
df1.loc[no_conflict_data, "age_conflict_interaction"] = np.nan

# Verify
print(df1[["conflict_index", "age_conflict_interaction"]].isnull().sum())
print(df1[df1["model_year"] < 2015]["conflict_index"].unique())  # should show nan

conflict_index              114
age_conflict_interaction    114
dtype: int64
[nan]


In [ ]:
# Rename mileage column if needed (adjust based on your actual column name)
if "mileage(km)" in df1.columns:
    df1.rename(columns={"mileage(km)": "mileage_km"}, inplace=True)

In [ ]:
# Log-transform target
df1["log_price"] = np.log1p(df1["price"])

In [ ]:
df1["transmission"] = df1["transmission"].replace({"9-Speed AT": "Automatic"})

In [ ]:
print(df1["mileage_km"].isna().sum())

0


**MODEL A**

In [ ]:
badge_flags = ["financing_available", "inspected", "full_service_history",
               "urgent_sale", "under_warranty"]

# Confirm mutual exclusivity before combining (safety check)
overlap_count = (df1[badge_flags].sum(axis=1) > 1).sum()
print(f"\nRows with 2+ badge flags simultaneously =1: {overlap_count} (expect 0)")

def get_badge(row):
    for flag in badge_flags:
        if row[flag] == 1:
            return flag
    return "No_Badge"

df1["listing_badge"] = df1[badge_flags].apply(get_badge, axis=1)
print("Listing badge category counts:")
print(df1["listing_badge"].value_counts())



Rows with 2+ badge flags simultaneously =1: 0 (expect 0)
Listing badge category counts:
listing_badge
No_Badge                834
under_warranty          595
urgent_sale             128
financing_available     105
inspected                98
full_service_history     37
Name: count, dtype: int64


In [ ]:
# Core numeric predictors
numeric_vars = ["car_age", "mileage_km"]

# Categorical controls (one-hot encoded)
categorical_vars = ["fuel_type", "transmission", "specs", "location", "export_options", "listing_badge" ]


# Combine all variables needed, drop rows with NaN in ANY of them
model_a_cols = ["log_price"] + numeric_vars + categorical_vars
model_a_cols = [c for c in model_a_cols if c in df1.columns]  # safety check

missing_cols = [c for c in (["log_price"] + numeric_vars + categorical_vars)
                if c not in df1.columns]
if missing_cols:
    print(f"\n⚠️  WARNING: These expected columns were not found: {missing_cols}")
    print(f"Available columns: {df1.columns.tolist()}")

df1_model_a = df1[model_a_cols].dropna().reset_index(drop=True)
print(f"\nModel A analytical sample: {len(df1_model_a)} rows "
      f"(dropped {len(df1) - len(df1_model_a)} rows with missing values in model variables)")



Model A analytical sample: 1797 rows (dropped 0 rows with missing values in model variables)


In [ ]:
model_a_predictors = ["mileage_km", "location", "fuel_type", "transmission", "specs",
                       "semiconductor_flag", "car_age", "export_options",
                       "listing_badge", "log_price"]
df1_model_a = df1[model_a_predictors]

In [ ]:
# drop_first=True avoids the dummy variable trap (multicollinearity from redundant category)
df1_model_a_encoded = pd.get_dummies(
    df1_model_a,
    columns=categorical_vars,
    drop_first=True,
    dtype=int  # ensures 0/1 not True/False
)

print(f"\nAfter one-hot encoding: {df1_model_a_encoded.shape[1]} columns")
print(f"Dummy variables created: {[c for c in df1_model_a_encoded.columns if c not in numeric_vars + ['log_price']]}")


After one-hot encoding: 34 columns
Dummy variables created: ['semiconductor_flag', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_Hybrid', 'fuel_type_Petrol', 'transmission_CVT', 'transmission_Manual', 'transmission_Not Mentioned', 'specs_Australian Specs', 'specs_Canadian Specs', 'specs_Chinese Specs', 'specs_European Specs', 'specs_GCC Specs', 'specs_Japanese Specs', 'specs_Korean Specs', 'specs_Not Sure', 'specs_Other Specs', 'location_Ajman', 'location_Al Ain', 'location_Dubai', 'location_Ras Al Khaimah', 'location_Sharjah', 'location_Unknown', 'export_options_Not Applicable', 'export_options_Not For Export', 'export_options_Only For Export', 'listing_badge_financing_available', 'listing_badge_full_service_history', 'listing_badge_inspected', 'listing_badge_under_warranty', 'listing_badge_urgent_sale']


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n" + "="*60)
print("  VIF CHECK — MODEL A PREDICTORS")
print("="*60)

# Select all predictor columns (exclude target)
predictor_cols = [c for c in df1_model_a_encoded.columns if c != "log_price"]
X_vif = df1_model_a_encoded[predictor_cols].astype(float)

# Add constant for VIF calculation
X_vif_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_with_const.values, i)
                    for i in range(X_vif_with_const.shape[1])]

vif_data = vif_data[vif_data["Feature"] != "const"].sort_values("VIF", ascending=False)
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data["VIF"] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️  WARNING: {len(high_vif)} variable(s) show VIF > 10:")
    print(high_vif.to_string(index=False))
    print("Consider reviewing these before trusting individual coefficients.")
else:
    print("\n✅ All predictors show VIF < 10 — no severe multicollinearity concern.")


  VIF CHECK — MODEL A PREDICTORS
                           Feature      VIF
      listing_badge_under_warranty      inf
           listing_badge_inspected      inf
         listing_badge_urgent_sale      inf
     export_options_Not Applicable      inf
 listing_badge_financing_available      inf
listing_badge_full_service_history      inf
                  fuel_type_Petrol 4.395761
                    location_Dubai 3.910888
                  location_Sharjah 3.766515
                   specs_GCC Specs 3.057409
                           car_age 2.599770
                  fuel_type_Hybrid 2.485715
                        mileage_km 2.386685
                  fuel_type_Diesel 2.255430
                fuel_type_Electric 2.221991
     export_options_Not For Export 2.152034
                  transmission_CVT 1.761344
                    location_Ajman 1.662644
                 specs_Other Specs 1.552649
              specs_European Specs 1.417959
            specs_Australian Specs 1.32769

In [ ]:
zero_var = X_vif.loc[:, X_vif.nunique() <= 1]
print("Zero-variance columns:")
print(zero_var.columns.tolist())

Zero-variance columns:
[]


In [ ]:
corr_matrix = X_vif.corr().abs()
high_corr = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_pairs = high_corr.stack().sort_values(ascending=False)
print(high_pairs.head(10))

mileage_km                     car_age                          0.704211
export_options_Not Applicable  listing_badge_under_warranty     0.654751
location_Dubai                 location_Sharjah                 0.631008
export_options_Not Applicable  export_options_Not For Export    0.595299
fuel_type_Hybrid               fuel_type_Petrol                 0.547020
fuel_type_Electric             fuel_type_Petrol                 0.493094
fuel_type_Diesel               fuel_type_Petrol                 0.442061
location_Dubai                 listing_badge_under_warranty     0.439544
export_options_Not For Export  listing_badge_under_warranty     0.389772
specs_GCC Specs                specs_Other Specs                0.373713
dtype: float64


In [ ]:
import numpy as np

rank = np.linalg.matrix_rank(X_vif_with_const.values)
n_cols = X_vif_with_const.shape[1]
print(f"Matrix rank: {rank}")
print(f"Number of columns: {n_cols}")
print(f"Rank deficiency: {n_cols - rank}")

Matrix rank: 33
Number of columns: 34
Rank deficiency: 1


In [ ]:
print("\n" + "="*60)
print("  MODEL A — BASELINE OLS REGRESSION RESULTS")
print("="*60 + "\n")

X = df1_model_a_encoded[predictor_cols].astype(float)
y = df1_model_a_encoded["log_price"].astype(float)
X = sm.add_constant(X)

model_a = sm.OLS(y, X).fit()
print(model_a.summary())


  MODEL A — BASELINE OLS REGRESSION RESULTS

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.448
Model:                            OLS   Adj. R-squared:                  0.438
Method:                 Least Squares   F-statistic:                     44.77
Date:                Wed, 01 Jul 2026   Prob (F-statistic):          1.44e-201
Time:                        11:47:00   Log-Likelihood:                -1692.2
No. Observations:                1797   AIC:                             3450.
Df Residuals:                    1764   BIC:                             3632.
Df Model:                          32                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [ ]:
import os
os.makedirs("regression_outputs", exist_ok=True)

# Save full summary as text
with open("regression_outputs/model_a_summary.txt", "w") as f:
    f.write(str(model_a.summary()))

# Save coefficients table as CSV (easy to paste into Word report)
coef_table = pd.DataFrame({
    "Variable": model_a.params.index,
    "Coefficient": model_a.params.values,
    "Std_Error": model_a.bse.values,
    "t_value": model_a.tvalues.values,
    "p_value": model_a.pvalues.values,
    "CI_lower": model_a.conf_int()[0].values,
    "CI_upper": model_a.conf_int()[1].values,
})
coef_table["Significant"] = coef_table["p_value"] < 0.05
coef_table.to_csv("regression_outputs/model_a_coefficients.csv", index=False)

print(f"\n\n{'='*60}")
print("  KEY METRICS SUMMARY")
print(f"{'='*60}")
print(f"R-squared:          {model_a.rsquared:.4f}")
print(f"Adjusted R-squared: {model_a.rsquared_adj:.4f}")
print(f"F-statistic:        {model_a.fvalue:.2f}  (p = {model_a.f_pvalue:.4g})")
print(f"AIC:                {model_a.aic:.2f}")
print(f"BIC:                {model_a.bic:.2f}")
print(f"N observations:     {int(model_a.nobs)}")

print(f"\nSaved: regression_outputs/model_a_summary.txt")
print(f"Saved: regression_outputs/model_a_coefficients.csv")

# Interpretation helper for car_age and mileage (your core research variables)
print(f"\n{'='*60}")
print("  QUICK INTERPRETATION — CORE VARIABLES")
print(f"{'='*60}")
if "car_age" in model_a.params.index:
    coef = model_a.params["car_age"]
    pct_effect = (np.exp(coef) - 1) * 100
    print(f"car_age:    coefficient = {coef:.4f}  →  each additional year of age "
          f"is associated with a {pct_effect:.2f}% {'decrease' if coef<0 else 'increase'} in price "
          f"(p = {model_a.pvalues['car_age']:.4g})")

if "mileage_km" in model_a.params.index:
    coef = model_a.params["mileage_km"]
    pct_effect_10k = (np.exp(coef * 10000) - 1) * 100
    print(f"mileage_km: coefficient = {coef:.6f}  →  each additional 10,000 km "
          f"is associated with a {pct_effect_10k:.2f}% {'decrease' if coef<0 else 'increase'} in price "
          f"(p = {model_a.pvalues['mileage_km']:.4g})")




  KEY METRICS SUMMARY
R-squared:          0.4482
Adjusted R-squared: 0.4382
F-statistic:        44.77  (p = 1.442e-201)
AIC:                3450.39
BIC:                3631.69
N observations:     1797

Saved: regression_outputs/model_a_summary.txt
Saved: regression_outputs/model_a_coefficients.csv

  QUICK INTERPRETATION — CORE VARIABLES
car_age:    coefficient = -0.0280  →  each additional year of age is associated with a -2.76% decrease in price (p = 0.0003394)
mileage_km: coefficient = -0.000005  →  each additional 10,000 km is associated with a -4.81% decrease in price (p = 1.092e-34)


In [ ]:
print(df1_model_a["specs"].value_counts())

specs
GCC Specs           1481
American Specs       143
Other Specs           52
European Specs        31
Canadian Specs        25
Korean Specs          19
Not Sure              17
Chinese Specs         12
Japanese Specs         9
Australian Specs       8
Name: count, dtype: int64


**MODEL B**

BASELINE + MACRO INTERACTION TERMS

(log_price ~ car_age + mileage_km + age_semiconductor_interaction
          + age_conflict_interaction + age_inflation_interaction + controls
)


Purpose: Test whether macro-sensitive depreciation effects (chip shortage,
regional conflict, inflation) add meaningful explanatory power beyond
Model A's baseline. mileage_oil_interaction is EXCLUDED (VIF 13.7-20.3,
confirmed unusable — see VIF comparison script results).

Compare against Model A using: Adjusted R², F-test, AIC/BIC.

In [ ]:
if "age_semiconductor_interaction" not in df1.columns or True:
    # semiconductor_flag is 0/1, so this interaction = car_age when flag=1, else 0
    df1["age_semiconductor_interaction"] = df1["car_age"] * df1["semiconductor_flag"]

df1["age_conflict_interaction"] = df1["car_age"] * df1["conflict_index"]
df1["age_inflation_interaction"] = df1["car_age"] * df1["inflation"]

print(f"\nage_conflict_interaction NaN: {df1['age_conflict_interaction'].isnull().sum()} (expect ~202)")
print(f"age_inflation_interaction NaN: {df1['age_inflation_interaction'].isnull().sum()} (expect 0, inflation has no NaN)")
print(f"age_semiconductor_interaction NaN: {df1['age_semiconductor_interaction'].isnull().sum()} (expect 0)")



age_conflict_interaction NaN: 114 (expect ~202)
age_inflation_interaction NaN: 0 (expect 0, inflation has no NaN)
age_semiconductor_interaction NaN: 0 (expect 0)


In [ ]:
# Core numeric predictors + macro interaction terms (mileage_oil EXCLUDED)
numeric_vars_B= ["car_age", "mileage_km",
                 "age_semiconductor_interaction",
                 "age_conflict_interaction",
                 "age_inflation_interaction"]

# Categorical controls (same as Model A, includes listing_badge fix)
categorical_vars_B = ["fuel_type", "transmission", "specs", "location",
                     "export_options", "listing_badge"]

model_b_cols = ["log_price"] + numeric_vars_B+ categorical_vars_B
model_b_cols = [c for c in model_b_cols if c in df1.columns]

missing_cols = [c for c in (["log_price"] + numeric_vars_B+ categorical_vars_B)
                if c not in df1.columns]
if missing_cols:
    print(f"\n⚠️  WARNING: These expected columns were not found: {missing_cols}")
    print(f"Available columns: {df1.columns.tolist()}")

df1_model_b = df1[model_b_cols].dropna().reset_index(drop=True)
print(f"\nModel B analytical sample: {len(df1_model_b)} rows "
      f"(dropped {len(df1) - len(df1_model_b)} rows with missing values in model variables)")
print("NOTE: larger row loss than Model A is EXPECTED — driven by conflict_index "
      "being unavailable for pre-2015 and 2026 model years.")



Model B analytical sample: 1683 rows (dropped 114 rows with missing values in model variables)
NOTE: larger row loss than Model A is EXPECTED — driven by conflict_index being unavailable for pre-2015 and 2026 model years.


In [ ]:
for col in categorical_vars_B:
    df1_model_b[col] = df1_model_b[col].astype(str)

df1_model_b_encoded = pd.get_dummies(
    df1_model_b,
    columns=categorical_vars_B,
    drop_first=True,
    dtype=int
)

print(f"\nAfter one-hot encoding: {df1_model_b_encoded.shape[1]} columns")


After one-hot encoding: 35 columns


In [ ]:
print("\n" + "="*60)
print("  VIF CHECK — MODEL B PREDICTORS")
print("="*60)

predictor_cols_b = [c for c in df1_model_b_encoded.columns if c != "log_price"]
X_vif_b = df1_model_b_encoded[predictor_cols_b].astype(float)
X_vif_b_const = sm.add_constant(X_vif_b, has_constant="add")

vif_data_b = pd.DataFrame()
vif_data_b["Feature"] = X_vif_b_const.columns
vif_data_b["VIF"] = [variance_inflation_factor(X_vif_b_const.values, i)
                      for i in range(X_vif_b_const.shape[1])]
vif_data_b = vif_data_b[vif_data_b["Feature"] != "const"].sort_values("VIF", ascending=False)
print(vif_data_b.to_string(index=False))

high_vif_b = vif_data_b[(vif_data_b["VIF"] > 10) & (~vif_data_b["VIF"].apply(np.isinf))]
if len(high_vif_b) > 0:
    print(f"\n⚠️  {len(high_vif_b)} variable(s) show VIF > 10 (excluding inf, per Model A's "
          f"known dummy-matrix artifact):")
    print(high_vif_b.to_string(index=False))
else:
    print("\n✅ Core numeric predictors (car_age, mileage_km, interaction terms) "
          "show acceptable VIF.")



  VIF CHECK — MODEL B PREDICTORS
                           Feature      VIF
         listing_badge_urgent_sale      inf
      listing_badge_under_warranty      inf
           listing_badge_inspected      inf
listing_badge_full_service_history      inf
     export_options_Not Applicable      inf
 listing_badge_financing_available      inf
                           car_age 5.604424
          age_conflict_interaction 4.449443
                  fuel_type_Petrol 4.184970
                    location_Dubai 3.964261
                  location_Sharjah 3.778013
                   specs_GCC Specs 3.120527
                  fuel_type_Hybrid 2.418992
                        mileage_km 2.239425
                fuel_type_Electric 2.226599
     export_options_Not For Export 2.215116
                  fuel_type_Diesel 1.984209
                  transmission_CVT 1.841531
                    location_Ajman 1.618298
                 specs_Other Specs 1.591107
              specs_European Specs 1.44367

In [ ]:
print("\n" + "="*60)
print("  MODEL B — MACRO-AUGMENTED OLS REGRESSION RESULTS")
print("="*60 + "\n")

X_b = df1_model_b_encoded[predictor_cols_b].astype(float)
y_b = df1_model_b_encoded["log_price"].astype(float)
X_b = sm.add_constant(X_b, has_constant="add")

model_b = sm.OLS(y_b, X_b).fit()
print(model_b.summary())




  MODEL B — MACRO-AUGMENTED OLS REGRESSION RESULTS

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.446
Model:                            OLS   Adj. R-squared:                  0.435
Method:                 Least Squares   F-statistic:                     40.17
Date:                Wed, 01 Jul 2026   Prob (F-statistic):          7.61e-185
Time:                        11:47:41   Log-Likelihood:                -1563.2
No. Observations:                1683   AIC:                             3194.
Df Residuals:                    1649   BIC:                             3379.
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------

In [ ]:
# NOTE: This is a NESTED model F-test, but only valid if Model A and Model B are run on the SAME sample.
#Since Model B drops ~202 extra rows (conflict_index NaN), Models A and B currently run on DIFFERENT samples (1,797 vs ~1,640ish).
# To compare fairly, Model A must be re-run on Model B's exact sample.This block does that automatically.

print("\n" + "="*60)
print("  MODEL A vs MODEL B — FAIR COMPARISON (SAME SAMPLE)")
print("="*60)

# Re-run Model A's variable set but restricted to Model B's row sample
model_a_vars_in_b_sample = ["car_age", "mileage_km"] + \
    [c for c in df1_model_b_encoded.columns if any(
        c.startswith(p) for p in ["fuel_type", "transmission", "specs",
                                    "location", "export_options", "listing_badge"])]

X_a_refit = df1_model_b_encoded[model_a_vars_in_b_sample].astype(float)
X_a_refit = sm.add_constant(X_a_refit, has_constant="add")
y_refit = df1_model_b_encoded["log_price"].astype(float)

model_a_refit = sm.OLS(y_refit, X_a_refit).fit()

print(f"\n{'Metric':<25}{'Model A (refit, same n)':<25}{'Model B':<25}")
print(f"{'-'*75}")
print(f"{'N observations':<25}{int(model_a_refit.nobs):<25}{int(model_b.nobs):<25}")
print(f"{'R-squared':<25}{model_a_refit.rsquared:<25.4f}{model_b.rsquared:<25.4f}")
print(f"{'Adj. R-squared':<25}{model_a_refit.rsquared_adj:<25.4f}{model_b.rsquared_adj:<25.4f}")
print(f"{'AIC':<25}{model_a_refit.aic:<25.2f}{model_b.aic:<25.2f}")
print(f"{'BIC':<25}{model_a_refit.bic:<25.2f}{model_b.bic:<25.2f}")

# Formal nested F-test: are the 3 added macro interaction terms jointly significant?
from statsmodels.stats.anova import anova_lm
try:
    anova_result = anova_lm(model_a_refit, model_b)
    print(f"\nNested F-test (H0: macro interaction terms add no explanatory power):")
    print(anova_result)
except Exception as e:
    print(f"\nNote: anova_lm requires identical row sets between models. "
          f"If this errors, compare AIC/BIC and Adj. R² manually instead. Error: {e}")

print("""
Interpretation guide:
- If Model B's Adjusted R² > Model A (refit)'s Adjusted R², macro interactions add value
- Lower AIC/BIC = better model (penalizes complexity)
- F-test p-value < 0.05 = the 3 added terms are jointly statistically significant
""")



  MODEL A vs MODEL B — FAIR COMPARISON (SAME SAMPLE)

Metric                   Model A (refit, same n)  Model B                  
---------------------------------------------------------------------------
N observations           1683                     1683                     
R-squared                0.4416                   0.4456                   
Adj. R-squared           0.4315                   0.4345                   
AIC                      3200.69                  3194.49                  
BIC                      3368.97                  3379.05                  

Nested F-test (H0: macro interaction terms add no explanatory power):
   df_resid         ssr  df_diff   ss_diff         F   Pr(>F)
0    1652.0  636.123792      0.0       NaN       NaN      NaN
1    1649.0  631.527935      3.0  4.595857  4.000123  0.00752

Interpretation guide:
- If Model B's Adjusted R² > Model A (refit)'s Adjusted R², macro interactions add value
- Lower AIC/BIC = better model (penalizes co

**MODEL C**

ERA-STRATIFIED REGRESSIONS

Runs the Model A specification (car_age + mileage_km + controls) SEPARATELY
for each of the 4 analytical eras, then compares the car_age coefficient
(depreciation rate) across eras.

Purpose: Directly answers the sub-question — does the rate of price
depreciation with age differ across distinct economic eras
(Pre-COVID, COVID, Semiconductor Shortage, Conflict-Driven)?

NOTE: Each era model uses a SMALLER sample than the pooled models (A/B),
so confidence intervals will be wider, especially for thinner eras
(Pre-COVID's Pre-Crash-Stable sub-segment = only 34 rows total in that era).
This is reported as a limitation, not a flaw.

In [ ]:
era_map = {
    "Pre-Crash Stable":      "Pre-COVID",
    "Oil Price Crash":       "Pre-COVID",
    "Pre-COVID":             "Pre-COVID",
    "COVID Period":          "COVID",
    "Shortage Peak":         "Semiconductor Shortage",
    "Recovery":              "Semiconductor Shortage",
    "Post-Recovery":         "Conflict-Driven",
    "Conflict Period 2026":  "Conflict-Driven",
}
df1["analytical_era"] = df1["market_period"].map(era_map)

print("\nAnalytical era distribution (before any row-dropping for model variables):")
print(df1["analytical_era"].value_counts())

unmapped = df1["analytical_era"].isnull().sum()
if unmapped > 0:
    print(f"\n⚠️  WARNING: {unmapped} rows did not map to any era — check market_period labels:")
    print(df1[df1["analytical_era"].isnull()]["market_period"].unique())



Analytical era distribution (before any row-dropping for model variables):
analytical_era
Semiconductor Shortage    586
Conflict-Driven           540
Pre-COVID                 395
COVID                     276
Name: count, dtype: int64


In [ ]:
numeric_vars_C= ["car_age", "mileage_km"]
categorical_vars_C = ["fuel_type", "transmission", "specs", "location",
                     "export_options", "listing_badge"]

model_cols = ["log_price", "analytical_era"] + numeric_vars_C + categorical_vars_C
model_cols = [c for c in model_cols if c in df1.columns]

df1_model_c = df1[model_cols].dropna().reset_index(drop=True)
print(f"\nModel C base sample (all eras combined, after dropna): {len(df1_model_c)} rows")
print(f"\nPer-era sample sizes after dropna:")
print(df1_model_c["analytical_era"].value_counts())


Model C base sample (all eras combined, after dropna): 1797 rows

Per-era sample sizes after dropna:
analytical_era
Semiconductor Shortage    586
Conflict-Driven           540
Pre-COVID                 395
COVID                     276
Name: count, dtype: int64


In [ ]:
era_order = ["Pre-COVID", "COVID", "Semiconductor Shortage", "Conflict-Driven"]
era_results = {}

print("\n" + "="*70)
print("  MODEL C — ERA-STRATIFIED REGRESSIONS")
print("="*70)

for era in era_order:
    print(f"\n{'─'*70}")
    print(f"  ERA: {era}")
    print(f"{'─'*70}")

    df_era = df1_model_c[df1_model_c["analytical_era"] == era].copy()
    n_rows = len(df_era)
    print(f"  Sample size: {n_rows}")

    if n_rows < 30:
        print(f"  ⚠️  WARNING: Very small sample ({n_rows} rows) — results will be unstable. Skipping regression.")
        continue

    # One-hot encode categoricals WITHIN this era's subset
    for col in categorical_vars:
        df_era[col] = df_era[col].astype(str)

    df_era_encoded = pd.get_dummies(df_era, columns=categorical_vars, drop_first=True, dtype=int)

    # Drop the analytical_era column itself
    predictor_cols_era = [c for c in df_era_encoded.columns if c not in ["log_price", "analytical_era"]]

    # Drop any column that's now constant within this era (zero variance)
    constant_cols = [c for c in predictor_cols_era if df_era_encoded[c].nunique() <= 1]
    if constant_cols:
        print(f"  Dropping {len(constant_cols)} zero-variance column(s) within this era: {constant_cols}")
        predictor_cols_era = [c for c in predictor_cols_era if c not in constant_cols]

    X_era = df_era_encoded[predictor_cols_era].astype(float)
    y_era = df_era_encoded["log_price"].astype(float)
    X_era = sm.add_constant(X_era, has_constant="add")

    try:
        model_era = sm.OLS(y_era, X_era).fit()
        era_results[era] = {
            "model": model_era,
            "n": n_rows,
            "car_age_coef": model_era.params.get("car_age", np.nan),
            "car_age_se": model_era.bse.get("car_age", np.nan),
            "car_age_pval": model_era.pvalues.get("car_age", np.nan),
            "car_age_ci_low": model_era.conf_int().loc["car_age", 0] if "car_age" in model_era.params.index else np.nan,
            "car_age_ci_high": model_era.conf_int().loc["car_age", 1] if "car_age" in model_era.params.index else np.nan,
            "mileage_coef": model_era.params.get("mileage_km", np.nan),
            "mileage_pval": model_era.pvalues.get("mileage_km", np.nan),
            "r_squared": model_era.rsquared,
            "adj_r_squared": model_era.rsquared_adj,
        }
        print(f"  car_age coefficient: {model_era.params.get('car_age', np.nan):.4f}  "
              f"(p = {model_era.pvalues.get('car_age', np.nan):.4g})")
        print(f"  mileage_km coefficient: {model_era.params.get('mileage_km', np.nan):.6f}  "
              f"(p = {model_era.pvalues.get('mileage_km', np.nan):.4g})")
        print(f"  R²: {model_era.rsquared:.4f}  |  Adj. R²: {model_era.rsquared_adj:.4f}")
    except Exception as e:
        print(f"  ❌ Regression failed for this era: {e}")


  MODEL C — ERA-STRATIFIED REGRESSIONS

──────────────────────────────────────────────────────────────────────
  ERA: Pre-COVID
──────────────────────────────────────────────────────────────────────
  Sample size: 395
  car_age coefficient: -0.0842  (p = 6.476e-07)
  mileage_km coefficient: -0.000003  (p = 6.327e-08)
  R²: 0.4663  |  Adj. R²: 0.4254

──────────────────────────────────────────────────────────────────────
  ERA: COVID
──────────────────────────────────────────────────────────────────────
  Sample size: 276
  car_age coefficient: -0.2027  (p = 0.008124)
  mileage_km coefficient: -0.000004  (p = 3.347e-05)
  R²: 0.4807  |  Adj. R²: 0.4219

──────────────────────────────────────────────────────────────────────
  ERA: Semiconductor Shortage
──────────────────────────────────────────────────────────────────────
  Sample size: 586
  car_age coefficient: 0.0211  (p = 0.7308)
  mileage_km coefficient: -0.000008  (p = 9.498e-16)
  R²: 0.4269  |  Adj. R²: 0.3992

────────────────

In [ ]:
print("\n\n" + "="*70)
print("  SUB-QUESTION ANSWER: DEPRECIATION RATE (car_age coefficient) BY ERA")
print("="*70)

comparison_rows = []
for era, res in era_results.items():
    pct_per_year = (np.exp(res["car_age_coef"]) - 1) * 100 if not np.isnan(res["car_age_coef"]) else np.nan
    comparison_rows.append({
        "Era": era,
        "N": res["n"],
        "car_age_coef": round(res["car_age_coef"], 4),
        "Approx_%_per_year": round(pct_per_year, 2),
        "p_value": round(res["car_age_pval"], 4),
        "CI_lower": round(res["car_age_ci_low"], 4),
        "CI_upper": round(res["car_age_ci_high"], 4),
        "Adj_R2": round(res["adj_r_squared"], 4),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

print("""
How to read this table:
- "Approx_%_per_year" = the approximate % price change per additional year of age in that era
- Compare CI_lower/CI_upper across eras — if they DON'T overlap between two eras,
  that's suggestive evidence the depreciation rate genuinely differs between them
- Wide CIs in thinner eras (small N) mean less certainty — report this as a limitation
""")




  SUB-QUESTION ANSWER: DEPRECIATION RATE (car_age coefficient) BY ERA
                   Era   N  car_age_coef  Approx_%_per_year  p_value  CI_lower  CI_upper  Adj_R2
             Pre-COVID 395       -0.0842              -8.07   0.0000   -0.1168   -0.0515  0.4254
                 COVID 276       -0.2027             -18.34   0.0081   -0.3522   -0.0531  0.4219
Semiconductor Shortage 586        0.0211               2.13   0.7308   -0.0991    0.1412  0.3992
       Conflict-Driven 540        0.0358               3.65   0.4145   -0.0503    0.1220  0.4892

How to read this table:
- "Approx_%_per_year" = the approximate % price change per additional year of age in that era
- Compare CI_lower/CI_upper across eras — if they DON'T overlap between two eras,
  that's suggestive evidence the depreciation rate genuinely differs between them
- Wide CIs in thinner eras (small N) mean less certainty — report this as a limitation



In [ ]:
import os
os.makedirs("regression_outputs", exist_ok=True)

comparison_df.to_csv("regression_outputs/model_c_era_comparison.csv", index=False)

for era, res in era_results.items():
    era_safe_name = era.replace(" ", "_")
    with open(f"regression_outputs/model_c_{era_safe_name}_summary.txt", "w") as f:
        f.write(str(res["model"].summary()))

print(f"\nSaved: regression_outputs/model_c_era_comparison.csv")
print(f"Saved: individual era summaries (one .txt file per era)")

print("""
Future Analysis Step:
Pooled robustness check — single regression with era dummies + car_age×era
interaction terms, to get a formal p-value for "is this era's slope
significantly different from the reference era." More rigorous but reintroduces
the age-era confound discussed earlier (each era only exists at certain ages
in 2026 cross-sectional data).
""")


Saved: regression_outputs/model_c_era_comparison.csv
Saved: individual era summaries (one .txt file per era)

Future Analysis Step:
Pooled robustness check — single regression with era dummies + car_age×era
interaction terms, to get a formal p-value for "is this era's slope
significantly different from the reference era." More rigorous but reintroduces
the age-era confound discussed earlier (each era only exists at certain ages
in 2026 cross-sectional data).



## Practical Implications and Conclusion

This analysis examined whether vehicle age, mileage, and the macroeconomic climate of a
car's manufacture year affect resale price in the UAE used-car market, and whether depreciation
rates differ across economic eras. Vehicle attributes remain the primary price driver (R² ≈ 0.44–0.45),
while macroeconomic factors — especially the semiconductor shortage and inflation — have a smaller
but statistically significant moderating effect.

### Recommendation for a UAE Dealership Group

- **Model A as a baseline valuation engine.** Age (−2.8%/year) and mileage (−4.8%/10,000 km)
  coefficients give a fast first-pass benchmark, letting a dealership flag listings priced well above
  or below model-predicted value and prioritize inspection effort toward genuine outliers.
- **Model B suggests a timing adjustment.** Vehicles from the 2021–2022 semiconductor shortage
  depreciate more slowly than age alone predicts (limited new-vehicle supply kept demand for older
  cars high) — these should get a smaller depreciation discount than standard age-based pricing tables suggest.
- **Model C is a direct pricing warning.** A vehicle should be priced off the depreciation curve of
  *its own* manufacturing era, not the current market average — applying the steep COVID-era curve
  (≈18.3%/year) to a car from a different era would systematically misprice it. Confidence is lower
  for the two most recent cohorts, so comparable-listing analysis should carry more weight there
  until more data accumulates.
- **Unexplained variance (~56%) is a scope boundary, not a weakness.** These models are a
  defensible statistical baseline — condition, brand, and trim-level detail still require dealer expertise
  to price any individual vehicle precisely.

### Limitations

- Model year serves as a proxy for listing date (unavailable from the source), so macro variables
  reflect production-year, not resale-moment, conditions
- All listings were scraped at a single point in time (mid-2026), creating a range-restriction problem:
  recent-era vehicles are necessarily young, preventing reliable depreciation-slope estimation for those cohorts
- 25 pre-2011 listings (1.25%) dropped for insufficient macro data; conflict-index data unavailable
  pre-2015 and for 2026, excluding 202 observations from Model B
- 2026 oil price is a partial-year estimate (Jan–Jun only)
- ~56% of price variation remains unexplained, likely due to brand, trim, accident history, and
  seller type — none captured in the scraped dataset

### Conclusion

UAE used-car pricing is fundamentally anchored in vehicle condition, with macroeconomic conditions
playing a secondary, era-specific moderating role. The most actionable output isn't a single pricing
formula, but a structured framework for knowing *when* standard depreciation assumptions can be
trusted — and when they can't.